# Telco Customer Churn Prediction

End-to-end churn model: EDA -> cleaning -> feature engineering -> baseline models ->
imbalance handling -> tuning -> evaluation -> save pipeline for deployment.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, RocCurveDisplay
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 2. Load Data

In [ ]:
df = pd.read_excel("Telco_customer_churn.xlsx")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Target Distribution

`Churn Value` (0/1) is the target we'll model on -- it's the numeric twin of `Churn Label`.
We check the class balance now because it determines how we train later (class_weight / SMOTE).

In [ ]:
df["Churn Label"].value_counts()

In [ ]:
df["Churn Label"].value_counts(normalize=True) * 100

**~73.5% stayed / ~26.5% churned.** This is a real imbalance -- a model that always predicts
"no churn" would score ~73.5% accuracy while being useless. We'll rely on ROC-AUC, precision/recall
and the confusion matrix instead of accuracy alone, and address the imbalance directly in modeling
(class_weight='balanced' vs SMOTE, compared side by side).

## 4. Missing Values Check

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

No nulls show up yet because `Total Charges` is loaded as a string (some blank values are
literal spaces, not NaN). We'll catch this in the cleaning step below.

## 5. Univariate Analysis

### Numerical features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["Monthly Charges"].hist(bins=30, ax=axes[0])
axes[0].set_title("Monthly Charges")
df["Tenure Months"].hist(bins=30, ax=axes[1])
axes[1].set_title("Tenure Months")
pd.to_numeric(df["Total Charges"], errors="coerce").hist(bins=30, ax=axes[2])
axes[2].set_title("Total Charges")
plt.tight_layout()
plt.show()

### Categorical features

In [ ]:
df["Contract"].value_counts().plot(kind="bar")
plt.title("Contract Type Distribution")
plt.show()

In [ ]:
df["Internet Service"].value_counts().plot(kind="bar")
plt.title("Internet Service Distribution")
plt.show()

## 6. Bivariate Analysis (vs Churn)

Every claim we write in the Business Insights section below has to trace back to a chart or
table in this section -- no unsupported claims.

In [ ]:
pd.crosstab(df["Contract"], df["Churn Label"], normalize="index") * 100

In [ ]:
df.boxplot(column="Tenure Months", by="Churn Label")
plt.title("Tenure by Churn")
plt.suptitle("")
plt.show()

In [ ]:
df.boxplot(column="Monthly Charges", by="Churn Label")
plt.title("Monthly Charges by Churn")
plt.suptitle("")
plt.show()

In [ ]:
pd.crosstab(df["Internet Service"], df["Churn Label"], normalize="index") * 100

## Business Insights

* Month-to-month contracts churn far more than one/two-year contracts (see crosstab above).
* Customers who churn skew toward lower tenure (see boxplot).
* Churned customers tend to have higher monthly charges (see boxplot).
* Fiber optic customers churn at a noticeably higher rate than DSL or no-internet customers
  (see Internet Service crosstab above) -- this one is now actually backed by a table,
  unlike the earlier draft.

## 7. Data Cleaning

### 7.1 Fix `Total Charges` dtype

In [ ]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Total Charges"].isnull().sum()

In [ ]:
df.loc[df["Total Charges"].isnull(), ["Tenure Months", "Monthly Charges", "Total Charges"]]

All the blanks belong to customers with **Tenure Months = 0**, i.e. brand-new customers who
haven't been billed yet. The correct fill value is **0**, not the median -- median would invent
billing history that doesn't exist for these customers.

In [ ]:
df["Total Charges"] = df["Total Charges"].fillna(0)

### 7.2 Drop leakage columns

In [ ]:
# Churn Reason: only known AFTER a customer has already churned -> leaks the answer.
# Churn Score / CLTV: pre-computed scores likely derived from a churn model / outcome -> leakage risk.
# Churn Label: the string twin of our numeric target -> duplicate target, must drop.
df = df.drop(columns=["Churn Reason", "Churn Score", "CLTV", "Churn Label"])

### 7.3 Drop identifier / zero-variance / no-signal columns

In [ ]:
df["Count"].value_counts()   # constant -> no signal

In [ ]:
df["Country"].value_counts()
df["State"].value_counts()   # constant (single country/state) -> no signal

### 7.4 Drop location columns

`City` one-hot-encodes into 1000+ near-unique columns, which blows up dimensionality, adds mostly
noise, and encourages overfitting on tiny per-city subsets. `Zip Code` / `Latitude` / `Longitude`
are just other encodings of the same location signal and have the same problem (plus they're not
meaningfully continuous/scalable). We drop all four rather than encoding around them.

In [ ]:
df = df.drop(columns=[
    "CustomerID", "Count", "Country", "State",
    "City", "Zip Code", "Lat Long", "Latitude", "Longitude"
])
df.shape

### 7.5 Duplicates

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
df.shape

## 8. Separate Features (X) and Target (y)

Target is `Churn Value` (0 = stayed, 1 = churned).

In [ ]:
X = df.drop(columns=["Churn Value"])
y = df["Churn Value"]

print(X.shape, y.shape)

In [ ]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

## 9. Train/Test Split

Split **before** any encoding/scaling is fit, so nothing about the test set leaks into
preprocessing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)
print(X_train.shape, X_test.shape)

## 10. Preprocessing Pipeline

Using `ColumnTransformer` + `Pipeline` instead of manual `get_dummies`/`StandardScaler` calls.
This does two things the original notebook didn't:
1. Guarantees preprocessing is fit only on training data and applied identically to test data / future data.
2. Bundles cleanly with the model so the *whole* thing (encoding + scaling + model) can be saved
   and reused in FastAPI/Streamlit at deployment time -- no separate encoder/scaler files to manage.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", "passthrough", categorical_cols),  # will be one-hot encoded via get_dummies below OR
    ],
    remainder="drop"
)

# Simpler / more transparent approach for this dataset size: one-hot encode with pandas first,
# then scale numeric columns only, inside a pipeline step. We use OneHotEncoder directly here
# so it's part of the sklearn Pipeline (fit on train, applied to test/production identically).
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
    ]
)

## 11. Baseline Models (class_weight='balanced')

First pass: default models, but with `class_weight='balanced'` (or `scale_pos_weight` for XGBoost)
so the imbalance is accounted for instead of ignored. We use 5-fold cross-validated ROC-AUC on the
training set (not a single train/test split) so the comparison isn't noisy.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

models_balanced = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(
        eval_metric="logloss", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE
    ),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, model in models_balanced.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = scores
    print(f"{name:22s}  mean ROC-AUC = {scores.mean():.4f}  (+/- {scores.std():.4f})")

### SMOTE comparison

In [ ]:
models_smote = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
}

cv_results_smote = {}

for name, model in models_smote.items():
    pipe = ImbPipeline([
        ("prep", preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", model),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results_smote[name] = scores
    print(f"{name:22s}  mean ROC-AUC = {scores.mean():.4f}  (+/- {scores.std():.4f})")

In [ ]:
comparison = pd.DataFrame({
    "Model": list(models_balanced.keys()),
    "ROC_AUC_class_weight": [cv_results[m].mean() for m in models_balanced],
    "ROC_AUC_SMOTE": [cv_results_smote[m].mean() for m in models_balanced],
}).sort_values("ROC_AUC_class_weight", ascending=False)

comparison

Pick whichever approach + model wins here for tuning below (edit the cell if a different
model/strategy comes out on top for your run -- this is written so it's not hardcoded to one
outcome).

## 12. Hyperparameter Tuning

Tuning the best-performing combination from the comparison above. Example shown for
XGBoost + class_weight-equivalent (`scale_pos_weight`) -- swap the estimator/param grid if a
different model won the comparison.

In [ ]:
best_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE)),
])

param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
}

search = RandomizedSearchCV(
    best_pipe,
    param_distributions=param_dist,
    n_iter=25,
    cv=cv,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Best CV ROC-AUC:", search.best_score_)
print("Best params:", search.best_params_)

final_model = search.best_estimator_

## 13. Final Evaluation on Held-Out Test Set

This is the step the original notebook skipped -- the tuned model actually gets scored here.

In [ ]:
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Stay", "Churn"], yticklabels=["Stay", "Churn"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix - Final Model")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(final_model, X_test, y_test)
plt.title("ROC Curve - Final Model")
plt.show()

### Threshold note

Default classification threshold is 0.5. For churn, missing an actual churner (false negative)
is usually more costly than a false alarm (false positive), since a missed churner is lost revenue
with no chance to intervene. If retention-campaign cost data is available, this is the point to
tune the decision threshold against a cost matrix rather than accepting 0.5 by default.

## 14. Feature Importance

In [ ]:
ohe = final_model.named_steps["prep"].named_transformers_["cat"]
feature_names = numeric_cols + list(ohe.get_feature_names_out(categorical_cols))

importances = final_model.named_steps["model"].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

feat_imp.plot(kind="barh", figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()

## 15. Save Pipeline for Deployment

Saves preprocessing + model together as one object -- this is what gets loaded in FastAPI/Streamlit,
so raw incoming records can be passed straight through `.predict()` without re-implementing encoding
logic separately.

In [ ]:
joblib.dump(final_model, "churn_pipeline.joblib")
print("Saved.")

## 16. Model Comparison Summary & Business Write-Up

Fill in after running:
- Final chosen model + CV ROC-AUC + test ROC-AUC
- Precision/recall tradeoff on the churn class and what that means operationally
  (e.g. "catches X% of churners at Y% false-alarm rate")
- Top drivers of churn from the feature importance chart, tied back to the EDA insights above
- Recommended next step (e.g. retention offer targeting month-to-month, high-monthly-charge,
  low-tenure fiber customers)